# Documentação Técnica — Projeto Case DataRisk

**Autor:** Renan Douglas Floriano Scavazzini  
**Data:** 2026-06-03  
**Repositório:** case_datarisk

---

## Sumário

1. Sumário Executivo  
2. Introdução  
3. Metodologia  
4. Definição da População  
5. Definição do Target  
6. Descrição das Bases de Dados  
7. Descrição das Features  
8. Análise Exploratória de Dados (EDA)  
9. Engenharia de Features  
10. Modelagem e Validação  
11. Política de Crédito e Regras de Negócio  
12. Conclusões e Recomendações  
13. Referências  
14. Apêndice (reprodutibilidade e artefatos)

---

## Sumário Executivo

Este documento consolida a entrega técnica do projeto `case_datarisk` e descreve, em nível profissional, as decisões, processos e artefatos produzidos para construir um score de crédito baseado em histórico de contratos e parcelas.

Pontos-chave:
- População-alvo: contratos com `status_contrato == 'Approved'`.  
- Target: indicador `ever_60` (atraso > 60 dias em qualquer parcela do contrato).  
- Pipeline: ingestão → limpeza → engenharia de features → treinamento (Logistic + LightGBM) → validação → política de crédito.  
- Métricas principais: ROC AUC, Gini, KS e stability checks (PSI).

Validação atual:
- LogisticRegression CV ROC AUC: 0.5717.  
- LightGBM CV ROC AUC: 0.9929 (accuracy 0.9908).

Risco e próximos passos: investigar vazamento detectado em validação porque o LightGBM apresenta desempenho muito superior ao baseline, revisar janelas temporais e features derivadas de parcelas futuras.

---

## 1. Introdução

Objetivo: projetar e validar um modelo de risco de crédito para decisões de aprovação de pedidos com base em dados internos de cadastro, histórico de contratos e histórico de parcelas. O projeto prioriza reprodutibilidade (ambiente, versões) e separação entre código reutilizável (`src/`) e notebooks que documentam as etapas.

Escopo: construção do dataset, definição de target, engenharia de features, treino e validação de modelos, e proposta de política de decisão.

---

## 2. Metodologia

Abordagem aplicada:
- Ingestão padronizada via funções em `src/data_loader.py`.  
- Construção da população de treino e score via `src/population.py`.  
- Target de inadimplência calculado a nível de contrato (ever_30/60/90 e `max_delay`) em `src/target.py`.  
- Features divididas em: perfil cadastral, agregados históricos por cliente, e atributos do contrato atual (módulo `src/feature_store.py`).  
- Preparação final do dataset com imputação e codificação em `src/feature_engineering.py`.  
- Treinamento: baseline `LogisticRegression` e modelo principal `LightGBM` (quando disponível).  
- Validação: Hold-out estratificado, cross-validation k-fold, métricas ROC AUC, Gini, KS, e checagens de vazamento e estabilidade.

Princípios de projeto:
- Reprodutibilidade: `requirements.txt` com versões pinned e `runtime.txt`.  
- Modularidade: lógica reutilizável em `src/` para facilitar testes e reuso.  
- Transparência: notebooks com markdown explicativo e funções documentadas.

---

## 3. Definição da População

Definição adotada: população elegível são contratos com `status_contrato == 'Approved'` (contratos efetivamente concedidos), uma vez que apenas estes permitem observar comportamento de pagamento e formação do target.

Detalhes operacionais:
- Fonte: `historico_emprestimos.parquet` (contratos) e `historico_parcelas.parquet` (parcelas).  
- Exclusões: contratos `Refused`, `Canceled` e `Unused offer` são removidos.  
- População de score: solicitações em `base_submissao.parquet` (todas as solicitações presentes no período de scoring).

Persistência: `data/processed/population_train.parquet` e `data/processed/population_score.parquet`.

---

## 4. Definição do Target

Target principal: `ever_60` que indica se, em qualquer parcela do contrato, o atraso real de pagamento (`data_real_pagamento - data_prevista_pagamento`) excedeu 60 dias.  

Rationale:
- `ever_60` é comumente utilizado como proxy de default de médio prazo e reduz ruído de pagamentos pontuais.  
- Também são calculadas versões `ever_30` e `ever_90` e `max_delay` para análises complementares.

Implementação técnica: agregação por `id_contrato` em `src/target.py` retornando colunas booleanas e `max_delay` inteiro.

---

## 5. Descrição das Bases de Dados

Principais fontes e colunas relevantes:
- `base_cadastral.parquet` — informações demográficas e socioeconômicas do cliente (`id_cliente`, `data_nascimento`, `renda_anual`, `qtd_membros_familia`, `nivel_educacao`, etc.).
- `historico_emprestimos.parquet` — contratos financeiros por `id_contrato` (`id_cliente`, `valor_credito`, `valor_parcela`, `qtd_parcelas_planejadas`, `status_contrato`, `data_decisao`, etc.).
- `historico_parcelas.parquet` — parcelas por contrato (`id_contrato`, `numero_parcela`, `data_prevista_pagamento`, `data_real_pagamento`, `valor_previsto_parcela`, `valor_pago_parcela`).
- `base_submissao.parquet` — solicitações de crédito para scoring (campos de proposta e `id_cliente`).

Observação sobre datas: convertemos todas as colunas de data relevantes para `datetime64[ns]` e validamos cobertura temporal (2017–2025).

---

## 6. Descrição das Features

As features são organizadas em três blocos principais:

1) Perfil cadastral (nivel estático):
- `idade` (anos), `renda_anual`, `qtd_membros_familia`, `qtd_filhos`, `estado_civil`, `nivel_educacao`, `possui_carro`, `possui_imovel`, `nota_regiao_cliente`.

2) Históricos agregados por cliente (comportamento):
- `customer_parcel_count`, `customer_avg_delay`, `customer_max_delay`, `customer_late30_rate`, `customer_late60_rate`, `customer_late90_rate`, `customer_payment_ratio` (valor pago / valor previsto), `customer_loan_count`, `customer_credit_sum`.

3) Atributos do contrato atual (contexto da solicitação):
- `valor_credito`, `valor_parcela`, `qtd_parcelas_planejadas`, `percentual_entrada`, `taxa_juros_padrao`, `tipo_produto`, `finalidade_emprestimo`, `area_venda`.

Features derivadas importantes:
- `loan_to_annual_income` = `valor_credito / renda_anual`  
- `installment_to_monthly_income` = `valor_parcela / (renda_anual/12)`  
- `income_per_family_member` = `renda_anual / qtd_membros_familia`  

Notas: todas as features derivadas e agregadas foram documentadas em `src/feature_store.py`.

---

## 7. Análise Exploratória de Dados (Resumo de ações sugeridas)

Para EDA executável, utilize o notebook `notebooks/01_population.ipynb` e siga os passos abaixo (resumo):

- Missing values: revisar colunas com alto missing% e documentar decisão (imputar, excluir ou criar categoria `missing`).
- Distribuições: plotar histograma/log-scale para `valor_credito`, `renda_anual`, `valor_parcela` e verificar outliers.  
- Atrasos: analisar a distribuição de `delay_days`, `max_delay` por contrato e as taxas `late30/60/90`.  
- Correlações: matriz de correlação para features numéricas e análise de multicolinearidade.  
- Time-split checks: garantir que features calculadas não utilizem informações de parcelas futuras ao período de corte (evitar vazamento).

Observação: resultados chave do EDA foram sumarizados no notebook principal e indicam cobertura temporal ampla e variáveis com ausência relevante (ex.: taxas de juros).

---

## 8. Engenharia de Features

Principais transformações aplicadas em `src/feature_engineering.py`:
- Imputação numérica por mediana para colunas selecionadas.  
- Categorias faltantes mapeadas para `missing` e codificadas com `OrdinalEncoder` (tratamento de `unknown_value=-1`).  
- Garantia de colunas presentes no dataset de score (criar colunas faltantes com valores neutros) para evitar erro na predição.  
- Criação e padronização de features derivadas descritas na seção anterior.

Boas práticas recomendadas:
- Manter o pipeline de pré-processamento reproduzível e idempotente.  
- Persistir a lista de features finais (`feature_list`) usada em treino para aplicar no scoring.

---

## 9. Modelagem e Validação

Estratégia de modelagem:
- Baseline: `LogisticRegression` (regularizada) para referência.  
- Modelo principal: `LightGBM` com hiperparâmetros iniciais (`n_estimators=200`, `random_state=42`).

Validação:
- Hold-out estratificado (25% test) para avaliações rápidas.  
- K-fold cross-validation estratificada (k=5) para estimativas robustas de variância das métricas.  
- Métricas: ROC AUC, Gini (2*AUC-1), KS, acurácia e análise de curvas de calibração.  
- Verificações adicionais: feature importance (gain / SHAP), PSI entre treino/score, e checagem de vazamento via inspeção das features mais importantes e análise temporal.

Resultados observados:
- LogisticRegression CV ROC AUC: 0.5717.  
- LightGBM CV ROC AUC: 0.9929.  
- LightGBM CV accuracy: 0.9908.  

Interpretação:
- A performance muito elevada do LightGBM indica que o modelo captura padrões fortes, mas também sugere potencial vazamento de dados ou informações futuras.  
- A baseline linear confirma que a separabilidade do problema é mais moderada, reforçando a necessidade de validação mais rigorosa.

Ações corretivas sugeridas:
- Validar janela temporal usada para construir features (garantir que nenhuma informação futura é usada).  
- Re-executar validação com k-fold por tempo (time-series split) se o problema for dependente de ordem temporal.  
- Rodar análise de importância por grupo (perfil vs. histórico vs. contrato) para identificar fontes de vazamento.

---

## 10. Política de Crédito e Regras de Negócio

Proposta básica (exemplo implementado em `src/policy.py`):
- Recebe `probability` (risco) e aplica `cutoff` para decisão binária (`Approve` / `Deny`).  
- Faixas de risco para monitoramento: Very Low, Low, Medium, High, Very High (bins definidos em `policy.build_credit_policy`).

Regras de negócio práticas sugeridas:
- Aplicar critérios adicionais (ex.: `loan_to_annual_income` máximo, `customer_late60_rate` tolerância) antes de negar automaticamente.  
- Para faixas `Medium`/`High` considerar análise manual ou políticas de mitigação (redução de prazo, aumento de entrada).

Governança:
- Documentar versão do modelo, data de treinamento e população usada.  
- Implementar monitoramento contínuo de performance (AUC, KS) e estabilidade (PSI) por coorte de tempo.

---

## 11. Conclusões e Recomendações

Resumo:
- Pipeline modular em `src/` e notebooks de documentação foram implementados.  
- A definição de população e target foi formalizada e persistida.  
- A validação cruzada mostrou `LogisticRegression CV ROC AUC 0.5717` e `LightGBM CV ROC AUC 0.9929`, indicando necessidade de investigação aprofundada de possível vazamento.

Recomendações imediatas:
1. Investigar vazamento: auditar features derivadas e repetir validação com splits temporais.  
2. Padronizar notebooks para garantir replicabilidade e reduzir diferenças entre treino/score.  
3. Implementar CV estratificado por tempo e análise de SHAP para explicar decisões do modelo.  
4. Preparar documentação final e plano de monitoramento para produção.

---

## 12. Referências

- Scikit-learn documentation — model selection and metrics.  
- LightGBM documentation — training and feature importance.  
- Handbooks and best practices for credit scoring (industry references).

---

## 13. Apêndice — Reprodutibilidade e Artefatos

Arquivos chave no repositório:
- `requirements.txt`, `runtime.txt` — versões do ambiente.  
- `src/` — módulos reutilizáveis: `data_loader.py`, `population.py`, `target.py`, `feature_store.py`, `feature_engineering.py`, `modeling.py`, `policy.py`.  
- `notebooks/` — notebooks com passos documentados: `01_population.ipynb`, `02_target_definition.ipynb`, ..., `07_credit_policy.ipynb`.  
- `data/processed/` — datasets persistidos usados em modelagem.  
- `outputs/models/` — modelos serializados (pkl).

Como reproduzir a pipeline localmente (exemplo):
1. Criar e ativar venv com Python 3.13.0.  
2. Instalar dependências: `pip install -r requirements.txt`.  
3. Executar `notebooks/01_population.ipynb` → `02_target_definition.ipynb` → ... na ordem.  
4. Modelos e submissões serão gerados em `outputs/`.

---